# 🌍 Geographic Analysis of Restaurant Locations

> **Task:** Plot the locations of restaurants on a map using longitude and latitude coordinates, and identify any spatial patterns or clusters of restaurants in specific areas.

---

**Domain:** Data Analytics · Geospatial Intelligence  
**Techniques:** Coordinate Mapping, DBSCAN Clustering, Convex Hull Boundary Detection  
**Dataset:** 9,551 restaurant records spanning 141 cities and 15 countries

---

## 📦 Cell 1 — Library Imports

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
from sklearn.cluster import DBSCAN
from scipy.spatial import ConvexHull

warnings.filterwarnings('ignore')
print('✔  All libraries imported successfully.')

✔  All libraries imported successfully.


## 📂 Cell 2 — Load & Inspect Dataset

In [ ]:
# ── Load raw data ──────────────────────────────────────────────────────────
df = pd.read_csv('../Dataset.csv')
print(f'Raw shape  : {df.shape}')
print(f'Columns    : {df.columns.tolist()}\n')
df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: 'geographic_analysis_project\\Dataset.csv'

## 🧹 Cell 3 — Data Cleaning & Validation

In [ ]:
# ── Remove sentinel (0,0) coordinates and out-of-range values ─────────────
df = df_raw.copy()
df = df[(df['Longitude'] != 0) & (df['Latitude'] != 0)]
df = df[(df['Longitude'].between(-180, 180)) & (df['Latitude'].between(-90, 90))]
df.dropna(subset=['Longitude', 'Latitude', 'City'], inplace=True)
df.reset_index(drop=True, inplace=True)

print('═' * 55)
print('  DATASET OVERVIEW')
print('═' * 55)
print(f'  Total records     : {len(df):,}')
print(f'  Unique cities     : {df["City"].nunique()}')
print(f'  Countries covered : {df["Country Code"].nunique()}')
print(f'  Latitude  range   : {df["Latitude"].min():.2f}° → {df["Latitude"].max():.2f}°')
print(f'  Longitude range   : {df["Longitude"].min():.2f}° → {df["Longitude"].max():.2f}°')
print('═' * 55)

# Null check
print('\nMissing values in key columns:')
print(df[['Longitude','Latitude','City','Cuisines','Aggregate rating']].isnull().sum())

## 📊 Cell 4 — City Distribution Analysis

In [ ]:
# ── Top cities and their restaurant counts ─────────────────────────────────
city_dist = df['City'].value_counts().head(20)
print('Top 20 Cities by Restaurant Count:')
print('─' * 40)
for city, count in city_dist.items():
    bar = '█' * (count // 100)
    print(f'  {city:<20} {count:>5}  {bar}')

# Market concentration
top3_share = city_dist.head(3).sum() / len(df) * 100
print(f'\n  Top-3 cities account for {top3_share:.1f}% of all records.')

## 🔬 Cell 5 — DBSCAN Spatial Clustering

In [ ]:
# ── DBSCAN with haversine metric (earth-surface distances) ─────────────────
# epsilon = 50 km expressed in radians (distance / earth_radius)
coords_rad = np.radians(df[['Latitude', 'Longitude']].values)

db = DBSCAN(
    eps       = 50 / 6371.0,   # 50 km radius
    min_samples = 30,          # minimum 30 restaurants to form a cluster
    algorithm = 'ball_tree',
    metric    = 'haversine'
)
df['Cluster'] = db.fit_predict(coords_rad)

n_clusters = len(set(df['Cluster'])) - (1 if -1 in df['Cluster'] else 0)
n_noise    = (df['Cluster'] == -1).sum()

print('DBSCAN Results:')
print(f'  Clusters identified : {n_clusters}')
print(f'  Noise / isolated    : {n_noise:,} points ({n_noise/len(df)*100:.1f}%)')

## 📋 Cell 6 — Cluster Summary Table

In [ ]:
# ── Aggregate per-cluster statistics ──────────────────────────────────────
cluster_stats = (
    df[df['Cluster'] != -1]
    .groupby('Cluster')
    .agg(
        Count      = ('Restaurant ID', 'count'),
        Avg_Rating = ('Aggregate rating', 'mean'),
        Avg_Votes  = ('Votes', 'mean'),
        Avg_Cost   = ('Average Cost for two', 'mean'),
        Top_City   = ('City', lambda x: x.value_counts().index[0]),
        Lat_Center = ('Latitude', 'mean'),
        Lon_Center = ('Longitude', 'mean'),
    )
    .sort_values('Count', ascending=False)
    .reset_index()
)
cluster_stats['Avg_Rating'] = cluster_stats['Avg_Rating'].round(2)
cluster_stats['Avg_Votes']  = cluster_stats['Avg_Votes'].round(0).astype(int)
cluster_stats['Lat_Center'] = cluster_stats['Lat_Center'].round(4)
cluster_stats['Lon_Center'] = cluster_stats['Lon_Center'].round(4)
cluster_stats

## 🍽️ Cell 7 — Cuisine & Rating Analysis

In [ ]:
# ── Top cuisine types globally ─────────────────────────────────────────────
print('Top 15 Cuisine Types (by restaurant count):')
cuisine_counts = (
    df['Cuisines'].dropna()
    .str.split(', ').explode().str.strip()
    .value_counts().head(15)
)
print(cuisine_counts.to_string())

print('\nAverage Rating by Price Range:')
print(df.groupby('Price range')['Aggregate rating'].mean().round(2).to_string())

print('\nOverall Rating Statistics:')
print(df['Aggregate rating'].describe().round(3).to_string())

## ▶️ Cell 8 — Run Full Visualisation Pipeline

Execute the complete analysis script to regenerate all 6 output charts:

In [ ]:
import subprocess, sys

script = os.path.join(os.path.dirname(os.getcwd()), 'notebooks', 'geographic_analysis.py')
result = subprocess.run([sys.executable, 'geographic_analysis.py'],
                        capture_output=True, text=True,
                        cwd=os.path.join(os.getcwd()))
print(result.stdout if result.stdout else result.stderr)

## 🗂️ Cell 9 — Output Gallery

The following charts are saved in the `outputs/` directory:

| File | Description |
|---|---|
| `01_global_distribution_map.png` | World scatter map coloured by aggregate rating |
| `02_dbscan_clustering_map.png` | DBSCAN cluster assignments on global map |
| `03_city_deepdive.png` | Per-city coordinate scatter with rating colour scale |
| `04_cluster_statistics_dashboard.png` | Multi-panel KPI & insight dashboard |
| `05_convex_hull_boundaries.png` | Convex hull boundaries for top-6 clusters |
| `06_executive_summary_infographic.png` | One-page executive summary infographic |
| `cluster_summary_table.csv` | Tabular cluster statistics for reporting |

---

## 📌 Key Findings

| # | Finding | Evidence |
|---|---|---|
| 1 | **South Asian NCR dominates globally** | New Delhi alone accounts for ~57% of all records |
| 2 | **4 distinct spatial clusters identified** | DBSCAN at ε = 50 km, min_samples = 30 |
| 3 | **Price tier correlates with rating** | Tier-4 restaurants average ~0.5 pts higher than Tier-1 |
| 4 | **North Indian cuisine leads by volume** | Followed by Chinese and Fast Food categories |
| 5 | **Secondary clusters in 3 continents** | Southeast Asia, Middle East, Europe, Oceania |
| 6 | **Urban core density** | Restaurant density mirrors commercial district geography |

---
*Analysis completed · Geographic Analysis Module · Data Analytics Internship Project*